In [2]:
# importing relevant packages

import pandas as pd
import numpy as np

In [5]:
# loading filtered time series window from step 4

df_ts_window = pd.read_csv("ts_window.csv")

df_ts_window

,stay_id,hr,map,rr,temp,gcs,lactate,creatinine,bilirubin,platelets,...,vasopressor_dose,fluid_input,ventilation_flag,sofa_resp,sofa_cardio,sofa_renal,sofa_liver,sofa_coag,sofa_cns,relative_hour
0,30000153,100.0,8.0,18.0,96.8,1.0,NaN,NaN,NaN,NaN,...,NaN,NaN,1,NaN,0.0,NaN,NaN,NaN,0.0,-0.150000
1,30000153,104.0,NaN,16.0,NaN,NaN,1.3,NaN,NaN,NaN,...,NaN,NaN,0,0.0,0.0,NaN,NaN,NaN,NaN,0.850000
2,30000153,83.0,NaN,16.0,99.1,NaN,2.1,NaN,NaN,NaN,...,NaN,NaN,0,0.0,NaN,NaN,NaN,NaN,NaN,1.850000
3,30000153,92.0,8.0,14.0,NaN,NaN,NaN,0.9,NaN,173.0,...,NaN,NaN,0,NaN,0.0,0.0,NaN,0.0,NaN,2.850000
4,30000153,83.0,NaN,16.0,99.5,1.0,NaN,NaN,NaN,NaN,...,NaN,NaN,0,0.0,0.0,NaN,NaN,NaN,0.0,3.850000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1011420,39999552,80.0,NaN,17.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,NaN,0.0,NaN,NaN,NaN,NaN,19.538056
1011421,39999552,72.0,NaN,15.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,NaN,0.0,NaN,NaN,NaN,NaN,20.538056
1011422,39999552,70.0,NaN,14.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,NaN,0.0,NaN,NaN,NaN,NaN,21.538056
1011423,39999552,80.0,NaN,17.0,98.2,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,NaN,0.0,NaN,NaN,NaN,NaN,22.538056


In [7]:
df_ts_window['stay_id'].nunique()

40457

In [10]:
#defining threshold for 12 clinical variables 

THRESHOLD_CONFIG = {"hr":         {"lower": 60,   "upper": 100,  "worst": "high"},
                    "rr":         {"lower": 12,   "upper": 20,   "worst": "high"},
                    "temp":       {"lower": 36.0, "upper": 38.0, "worst": "high"},
                    "map":        {"lower": 65,   "upper": 110,  "worst": "low"},
                    "gcs":        {"lower": 9,    "upper": None, "worst": "low"},
                    "lactate":    {"lower": None, "upper": 2.0,  "worst": "high"},
                    "creatinine": {"lower": None, "upper": 1.2,  "worst": "high"},
                    "bilirubin":  {"lower": None, "upper": 1.2,  "worst": "high"},
                    "platelets":  {"lower": 150,  "upper": None, "worst": "low"},
                    "wbc":        {"lower": 4.0,  "upper": 12.0, "worst": "high"},
                    "sodium":     {"lower": 136,  "upper": 145,  "worst": "low"},
                    "bun":        {"lower": None, "upper": 20,   "worst": "high"}}

In [ ]:
# defining all variables and 6-hour bins of the observation window
# bins: [-3, 6h], [6, 12h], [12, 18h], [18, 24h]

ALL_VARS = [
    "hr", "rr", "ventilation_flag", "urine_output", "sofa_cardio",
    "gcs", "temp", "sofa_cns", "sofa_renal", "sodium", "creatinine",
    "bun", "platelets", "vasopressor_dose", "sofa_coag", "map",
    "lactate", "sofa_resp", "bilirubin", "sofa_liver", "fluid_input", "wbc"
]

BIN_EDGES  = [-3, 6, 12, 18, 24]
BIN_LABELS = ["0_6h", "6_12h", "12_18h", "18_24h"]

All variables : 22
Bin labels    : ['0_6h', '6_12h', '12_18h', '18_24h']


In [15]:
# assigning each row to a 6-hour bin based on relative_hour
# Reference: https://pandas.pydata.org/docs/reference/api/pandas.cut.html

df_ts_window["bin"] = pd.cut(
    df_ts_window["relative_hour"],
    bins=BIN_EDGES,
    labels=BIN_LABELS,
    right=True,
    include_lowest=True
)

print(df_ts_window["bin"].value_counts().sort_index())

bin
0_6h      283199
6_12h     242742
12_18h    242742
18_24h    242742
Name: count, dtype: int64


In [23]:
# counting measurements per 6-hour bin for heart rate (hr)
# count = number of measurements in that bin
# flag  = 1 if at least one measurement exists, 0 otherwise

hr_bin = df_ts_window.dropna(subset=["hr"]).groupby(["stay_id", "bin"]).size().unstack(fill_value=0)
hr_bin = hr_bin.reindex(columns=BIN_LABELS, fill_value=0)

for label in BIN_LABELS:
    hr_bin[f"hr_count_{label}"] = hr_bin[label]
    hr_bin[f"hr_flag_{label}"]  = (hr_bin[label] > 0).astype(int)

hr_bin = hr_bin.drop(columns=BIN_LABELS).reset_index()
hr_bin

bin,stay_id,hr_count_0_6h,hr_flag_0_6h,hr_count_6_12h,hr_flag_6_12h,hr_count_12_18h,hr_flag_12_18h,hr_count_18_24h,hr_flag_18_24h
0,30000153,7,1,6,1,6,1,6,1
1,30000646,7,1,6,1,6,1,6,1
2,30001148,4,1,6,1,6,1,6,1
3,30001336,7,1,5,1,6,1,6,1
4,30001396,6,1,6,1,6,1,6,1
...,...,...,...,...,...,...,...,...,...
40384,39999172,7,1,6,1,6,1,6,1
40385,39999230,7,1,6,1,6,1,6,1
40386,39999286,7,1,6,1,6,1,6,1
40387,39999384,5,1,6,1,6,1,6,1


In [19]:
# getting all stay_ids for reindexing
# some stays may have no measurements for certain variables
# reindex ensures all 40,457 stays appear with 0 for missing bins

all_stay_ids = sorted(df_ts_window["stay_id"].unique())

print(f"Total stays: {len(all_stay_ids)}")

Total stays: 40457


In [24]:
# defining function to compute count and flag per 6-hour bin for one variable
# reindexing to include all stays even if they have no measurements
# Reference: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reindex.html

def compute_bin_features(var):
    sub = df_ts_window.dropna(subset=[var]).groupby(["stay_id", "bin"]).size().unstack(fill_value=0)
    sub = sub.reindex(index=all_stay_ids, fill_value=0)
    sub = sub.reindex(columns=BIN_LABELS, fill_value=0)

    result = pd.DataFrame(index=all_stay_ids)
    for label in BIN_LABELS:
        result[f"{var}_count_{label}"] = sub[label]
        result[f"{var}_flag_{label}"]  = (sub[label] > 0).astype(int)

    return result

In [26]:
# computing bin features for all variables
# Reference: https://pandas.pydata.org/docs/reference/api/pandas.concat.html

bin_frames = []
for var in ALL_VARS:
    if var in df_ts_window.columns:
        bin_frames.append(compute_bin_features(var))
        print(f"Done: {var}")

df_bin = pd.concat(bin_frames, axis=1).reset_index()
df_bin = df_bin.rename(columns={"index": "stay_id"})

print(f"\nShape: {df_bin.shape}")
df_bin.head()

Done: hr
Done: rr
Done: ventilation_flag
Done: urine_output
Done: sofa_cardio
Done: gcs
Done: temp
Done: sofa_cns
Done: sofa_renal
Done: sodium
Done: creatinine
Done: bun
Done: platelets
Done: vasopressor_dose
Done: sofa_coag
Done: map
Done: lactate
Done: sofa_resp
Done: bilirubin
Done: sofa_liver
Done: fluid_input
Done: wbc

Shape: (40457, 177)


/var/folders/gs/w9gkn6md2jj51cc9mwcqwpnr0000gn/T/ipykernel_37474/2066471157.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_bin = pd.concat(bin_frames, axis=1).reset_index()


,stay_id,hr_count_0_6h,hr_flag_0_6h,hr_count_6_12h,hr_flag_6_12h,hr_count_12_18h,hr_flag_12_18h,hr_count_18_24h,hr_flag_18_24h,rr_count_0_6h,...,fluid_input_count_18_24h,fluid_input_flag_18_24h,wbc_count_0_6h,wbc_flag_0_6h,wbc_count_6_12h,wbc_flag_6_12h,wbc_count_12_18h,wbc_flag_12_18h,wbc_count_18_24h,wbc_flag_18_24h
0,30000153,7,1,6,1,6,1,6,1,7,...,0,0,0,0,0,0,0,0,0,0
1,30000646,7,1,6,1,6,1,6,1,7,...,0,0,0,0,0,0,0,0,0,0
2,30001148,4,1,6,1,6,1,6,1,4,...,0,0,0,0,0,0,0,0,0,0
3,30001336,7,1,5,1,6,1,6,1,7,...,0,0,0,0,0,0,0,0,0,0
4,30001396,6,1,6,1,6,1,6,1,6,...,0,0,0,0,0,0,0,0,0,0


In [27]:
# computing time to first abnormal value for heart rate (hr)
# abnormal = below lower threshold OR above upper threshold
# NaN if no abnormal value occurs during observation window
# Reference: step3_timeseries_advanced.py from group pipeline

sub = df_ts_window[["stay_id", "relative_hour", "hr"]].dropna(subset=["hr"])

# mark abnormal rows (hr < 60 or hr > 100)
abnormal_mask = (sub["hr"] < 60) | (sub["hr"] > 100)

# get first abnormal time per stay
hr_first_abnormal = sub[abnormal_mask].groupby("stay_id")["relative_hour"].min()
hr_first_abnormal.name = "hr_time_to_first_abnormal_hours"

hr_first_abnormal = hr_first_abnormal.reindex(all_stay_ids)

print(hr_first_abnormal.describe())

count    27235.000000
mean         5.228745
std          6.644394
min         -0.998889
25%         -0.033333
50%          2.333333
75%          8.815278
max         24.000000
Name: hr_time_to_first_abnormal_hours, dtype: float64


In [28]:
# computing time to first abnormal for all 12 threshold variables
# Reference: step3_timeseries_advanced.py from group pipeline

first_abnormal_frames = []

for var, cfg in THRESHOLD_CONFIG.items():
    if var not in df_ts_window.columns:
        continue

    sub = df_ts_window[["stay_id", "relative_hour", var]].dropna(subset=[var])

    # build abnormal mask based on thresholds
    mask = pd.Series(False, index=sub.index)
    if cfg["lower"] is not None:
        mask = mask | (sub[var] < cfg["lower"])
    if cfg["upper"] is not None:
        mask = mask | (sub[var] > cfg["upper"])

    first = sub[mask].groupby("stay_id")["relative_hour"].min()
    first.name = f"{var}_time_to_first_abnormal_hours"
    first_abnormal_frames.append(first.reindex(all_stay_ids))

    print(f"Done: {var}")

df_first_abnormal = pd.concat(first_abnormal_frames, axis=1)
df_first_abnormal.index.name = "stay_id"
df_first_abnormal = df_first_abnormal.reset_index()

print(f"\nShape: {df_first_abnormal.shape}")
df_first_abnormal.head()

Done: hr
Done: rr
Done: temp
Done: map
Done: gcs
Done: lactate
Done: creatinine
Done: bilirubin
Done: platelets
Done: wbc
Done: sodium
Done: bun

Shape: (40457, 13)


,stay_id,hr_time_to_first_abnormal_hours,rr_time_to_first_abnormal_hours,temp_time_to_first_abnormal_hours,map_time_to_first_abnormal_hours,gcs_time_to_first_abnormal_hours,lactate_time_to_first_abnormal_hours,creatinine_time_to_first_abnormal_hours,bilirubin_time_to_first_abnormal_hours,platelets_time_to_first_abnormal_hours,wbc_time_to_first_abnormal_hours,sodium_time_to_first_abnormal_hours,bun_time_to_first_abnormal_hours
0,30000153,0.850000,6.850000,-0.150000,-0.150000,-0.150000,1.85,NaN,NaN,NaN,NaN,NaN,2.85
1,30000646,0.343889,-0.656111,-0.656111,NaN,-0.656111,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,30001148,NaN,2.816944,2.816944,2.816944,2.816944,NaN,NaN,NaN,NaN,NaN,1.816944,NaN
3,30001336,1.253333,-0.746667,-0.746667,NaN,7.253333,NaN,NaN,NaN,NaN,NaN,-0.746667,NaN
4,30001396,NaN,3.200000,0.200000,NaN,0.200000,2.20,15.2,NaN,NaN,NaN,NaN,15.20


In [29]:
# computing time to nadir (worst value) for all 12 threshold variables
# worst direction per variable: "high" = idxmax, "low" = idxmin
# Reference: step3_timeseries_advanced.py from group pipeline

nadir_frames = []

for var, cfg in THRESHOLD_CONFIG.items():
    if var not in df_ts_window.columns:
        continue

    sub = df_ts_window[["stay_id", "relative_hour", var]].dropna(subset=[var])

    if cfg["worst"] == "high":
        idx = sub.groupby("stay_id")[var].idxmax()
    else:
        idx = sub.groupby("stay_id")[var].idxmin()

    nadir_times = sub.loc[idx].set_index("stay_id")["relative_hour"]
    nadir_times.name = f"{var}_time_to_nadir_hours"
    nadir_frames.append(nadir_times.reindex(all_stay_ids))

    print(f"Done: {var}")

df_nadir = pd.concat(nadir_frames, axis=1)
df_nadir.index.name = "stay_id"
df_nadir = df_nadir.reset_index()

print(f"\nShape: {df_nadir.shape}")
df_nadir.head()

Done: hr
Done: rr
Done: temp
Done: map
Done: gcs
Done: lactate
Done: creatinine
Done: bilirubin
Done: platelets
Done: wbc
Done: sodium
Done: bun

Shape: (40457, 13)


,stay_id,hr_time_to_nadir_hours,rr_time_to_nadir_hours,temp_time_to_nadir_hours,map_time_to_nadir_hours,gcs_time_to_nadir_hours,lactate_time_to_nadir_hours,creatinine_time_to_nadir_hours,bilirubin_time_to_nadir_hours,platelets_time_to_nadir_hours,wbc_time_to_nadir_hours,sodium_time_to_nadir_hours,bun_time_to_nadir_hours
0,30000153,7.850000,8.850000,7.850000,4.850000,-0.150000,1.850000,14.850000,NaN,14.850000,NaN,0.850000,2.850000
1,30000646,0.343889,0.343889,4.343889,NaN,-0.656111,4.343889,4.343889,4.343889,17.343889,NaN,0.343889,0.343889
2,30001148,2.816944,18.816944,20.816944,2.816944,2.816944,1.816944,3.816944,NaN,1.816944,NaN,1.816944,14.816944
3,30001336,18.253333,23.253333,-0.746667,NaN,7.253333,NaN,-0.746667,-0.746667,5.253333,NaN,-0.746667,12.253333
4,30001396,3.200000,9.200000,12.200000,NaN,0.200000,15.200000,15.200000,2.200000,15.200000,NaN,15.200000,15.200000


In [30]:
# computing threshold exposure and crossings for heart rate (hr)
# exposure = total hours outside normal range using LOCF
# crossings = number of transitions into abnormal state
# Reference: step3_timeseries_advanced.py from group pipeline

sub = df_ts_window[["stay_id", "relative_hour", "hr"]].dropna(subset=["hr"]).copy()
sub = sub.sort_values(["stay_id", "relative_hour"])

# LOCF: each value persists until next measurement or end of window (hour 24)
next_hour = sub.groupby("stay_id")["relative_hour"].shift(-1)
sub["duration"] = (next_hour.fillna(24) - sub["relative_hour"]).clip(lower=0)

# below lower threshold (hr < 60)
sub["is_below"] = sub["hr"] < 60
sub["below_dur"] = sub["duration"] * sub["is_below"].astype(float)
hr_time_below = sub.groupby("stay_id")["below_dur"].sum()

prev_below = sub.groupby("stay_id")["is_below"].shift(1)
sub["cross_lower"] = sub["is_below"] & (~prev_below.fillna(False))
hr_cross_lower = sub.groupby("stay_id")["cross_lower"].sum()

# above upper threshold (hr > 100)
sub["is_above"] = sub["hr"] > 100
sub["above_dur"] = sub["duration"] * sub["is_above"].astype(float)
hr_time_above = sub.groupby("stay_id")["above_dur"].sum()

prev_above = sub.groupby("stay_id")["is_above"].shift(1)
sub["cross_upper"] = sub["is_above"] & (~prev_above.fillna(False))
hr_cross_upper = sub.groupby("stay_id")["cross_upper"].sum()

print("hr_time_below_lower_hours:")
print(hr_time_below.describe())
print("\nhr_time_above_upper_hours:")
print(hr_time_above.describe())

hr_time_below_lower_hours:
count    40389.000000
mean         1.403287
std          3.936213
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         24.974722
Name: below_dur, dtype: float64

hr_time_above_upper_hours:
count    40389.000000
mean         4.425677
std          6.976804
min          0.000000
25%          0.000000
50%          0.000000
75%          6.000000
max         24.996111
Name: above_dur, dtype: float64


In [31]:
# computing threshold exposure and crossings for all 12 threshold variables
# Reference: step3_timeseries_advanced.py from group pipeline

exposure_frames = []

for var, cfg in THRESHOLD_CONFIG.items():
    if var not in df_ts_window.columns:
        continue

    sub = df_ts_window[["stay_id", "relative_hour", var]].dropna(subset=[var]).copy()
    sub = sub.sort_values(["stay_id", "relative_hour"])

    # LOCF duration
    next_hour = sub.groupby("stay_id")["relative_hour"].shift(-1)
    sub["duration"] = (next_hour.fillna(24) - sub["relative_hour"]).clip(lower=0)

    result = pd.DataFrame(index=all_stay_ids)
    result.index.name = "stay_id"

    if cfg["lower"] is not None:
        sub["is_below"] = sub[var] < cfg["lower"]
        sub["below_dur"] = sub["duration"] * sub["is_below"].astype(float)
        result[f"{var}_time_below_lower_hours"] = sub.groupby("stay_id")["below_dur"].sum().reindex(all_stay_ids)

        prev_below = sub.groupby("stay_id")["is_below"].shift(1)
        sub["cross_lower"] = sub["is_below"] & (~prev_below.fillna(False))
        result[f"{var}_n_crossings_lower"] = sub.groupby("stay_id")["cross_lower"].sum().reindex(all_stay_ids)

    if cfg["upper"] is not None:
        sub["is_above"] = sub[var] > cfg["upper"]
        sub["above_dur"] = sub["duration"] * sub["is_above"].astype(float)
        result[f"{var}_time_above_upper_hours"] = sub.groupby("stay_id")["above_dur"].sum().reindex(all_stay_ids)

        prev_above = sub.groupby("stay_id")["is_above"].shift(1)
        sub["cross_upper"] = sub["is_above"] & (~prev_above.fillna(False))
        result[f"{var}_n_crossings_upper"] = sub.groupby("stay_id")["cross_upper"].sum().reindex(all_stay_ids)

    exposure_frames.append(result)
    print(f"Done: {var}")

df_exposure = pd.concat(exposure_frames, axis=1)
df_exposure.index.name = "stay_id"
df_exposure = df_exposure.reset_index()

print(f"\nShape: {df_exposure.shape}")
df_exposure.head()

Done: hr
Done: rr
Done: temp
Done: map
Done: gcs
Done: lactate
Done: creatinine
Done: bilirubin
Done: platelets
Done: wbc
Done: sodium
Done: bun

Shape: (40457, 37)


,stay_id,hr_time_below_lower_hours,hr_n_crossings_lower,hr_time_above_upper_hours,hr_n_crossings_upper,rr_time_below_lower_hours,rr_n_crossings_lower,rr_time_above_upper_hours,rr_n_crossings_upper,temp_time_below_lower_hours,...,wbc_time_below_lower_hours,wbc_n_crossings_lower,wbc_time_above_upper_hours,wbc_n_crossings_upper,sodium_time_below_lower_hours,sodium_n_crossings_lower,sodium_time_above_upper_hours,sodium_n_crossings_upper,bun_time_above_upper_hours,bun_n_crossings_upper
0,30000153,0.0,0.0,16.0,16.0,5.15,6.0,3.000000,3.0,0.0,...,NaN,NaN,NaN,NaN,0.000000,0.0,0.0,0.0,21.15,2.0
1,30000646,0.0,0.0,1.0,1.0,0.00,0.0,15.000000,15.0,0.0,...,NaN,NaN,NaN,NaN,0.000000,0.0,0.0,0.0,0.00,0.0
2,30001148,0.0,0.0,0.0,0.0,4.00,4.0,2.183056,3.0,0.0,...,NaN,NaN,NaN,NaN,2.000000,1.0,0.0,0.0,0.00,0.0
3,30001336,7.0,6.0,0.0,0.0,0.00,0.0,20.746667,21.0,0.0,...,NaN,NaN,NaN,NaN,24.746667,3.0,0.0,0.0,0.00,0.0
4,30001396,0.0,0.0,0.0,0.0,0.00,0.0,18.000000,18.0,0.0,...,NaN,NaN,NaN,NaN,0.000000,0.0,0.0,0.0,8.80,1.0


In [32]:
# merging all advanced features together
# Reference: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html

df_advanced = df_bin.merge(df_first_abnormal, on="stay_id", how="left")
df_advanced = df_advanced.merge(df_nadir, on="stay_id", how="left")
df_advanced = df_advanced.merge(df_exposure, on="stay_id", how="left")

print(f"Final shape: {df_advanced.shape}")
df_advanced.head()

Final shape: (40457, 237)


,stay_id,hr_count_0_6h,hr_flag_0_6h,hr_count_6_12h,hr_flag_6_12h,hr_count_12_18h,hr_flag_12_18h,hr_count_18_24h,hr_flag_18_24h,rr_count_0_6h,...,wbc_time_below_lower_hours,wbc_n_crossings_lower,wbc_time_above_upper_hours,wbc_n_crossings_upper,sodium_time_below_lower_hours,sodium_n_crossings_lower,sodium_time_above_upper_hours,sodium_n_crossings_upper,bun_time_above_upper_hours,bun_n_crossings_upper
0,30000153,7,1,6,1,6,1,6,1,7,...,NaN,NaN,NaN,NaN,0.000000,0.0,0.0,0.0,21.15,2.0
1,30000646,7,1,6,1,6,1,6,1,7,...,NaN,NaN,NaN,NaN,0.000000,0.0,0.0,0.0,0.00,0.0
2,30001148,4,1,6,1,6,1,6,1,4,...,NaN,NaN,NaN,NaN,2.000000,1.0,0.0,0.0,0.00,0.0
3,30001336,7,1,5,1,6,1,6,1,7,...,NaN,NaN,NaN,NaN,24.746667,3.0,0.0,0.0,0.00,0.0
4,30001396,6,1,6,1,6,1,6,1,6,...,NaN,NaN,NaN,NaN,0.000000,0.0,0.0,0.0,8.80,1.0


In [33]:
# exporting advanced time series features
# Reference: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html

df_advanced.to_csv("cohort_step_5.csv", index=False)

print(f"Saved: {df_advanced.shape[0]:,} rows x {df_advanced.shape[1]} columns")

Saved: 40,457 rows x 237 columns
